In [2]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [3]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; OH it's GEMINI !!!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


An API key was found, but it doesn't start sk-proj-; OH it's GEMINI !!!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [4]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message =  "Hello Gemini! I am using you via the OpenAI" 

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user', 'content': 'Hello Gemini! I am using you via the OpenAI'}]

In [5]:
gemini = OpenAI(base_url= "https://generativelanguage.googleapis.com/v1beta/openai/",api_key=api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
response.choices[0].message.content

'Actually, there seems to be a slight mix-up! \n\nI am **Gemini**, a large language model trained by **Google**. \n\nIf you are currently interacting with me, you are using a Google service or platform (like gemini.google.com). **OpenAI** is a separate company that created ChatGPT. We are two different AI models developed by different organizations. \n\nHow can I help you today?'

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [6]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
"You are an adult comedy assistant. Rewrite website content with a sharp, irreverent, 
profanity-laced sense of humor. Keep it witty, sarcastic, and entertaining while preserving 
the original meaning and key information. Use swearing for comedic effect, not gratuitously, 
and avoid hate, harassment, or encouraging harm. Wrap text in markdown only. Output only valid Markdown. 
Do not include code fences."
"""

In [7]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website in the tone mentioned in the system prompt.
If it includes news or announcements, then exclude them.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [8]:
model_gem = "gemini-3.1-flash-lite"

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = gemini.chat.completions.create(model=model_gem, messages=messages)
response.choices[0].message.content

'2 + 2 = 4'

## And now let's build useful messages for GPT-4.1-mini, using a function

In [10]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [11]:

lit = fetch_website_contents("https://iskcon.org/")
print(lit)

ISKCON | The Official Website of the International Society for Krishna Consciousness

The official site of the
International Society for
Krishna Consciousness
Founder
-Acharya
His Divine Grace
A.C. Bhaktivedanta Swami Prabhupada
Home
About
Locate a Center
Mission & Vision
What is ISKCON?
What is Krishna Consciousness?
His Divine Grace A.C. Bhaktivedanta Swami Prabhupada
Governance
Ministries
BBT
History
A Global Family
FAQ
Experience
Kirtan
Festivals
Sunday Feast/Open House
Eco-communities
Arts & Culture
Wisdom
What is Bhakti-yoga
Hare Krishna Maha-Mantra
Studying Sacred Texts
God is a Person
Karma
Reincarnation
Vegetarianism
Social Action
Volunteer
Food Relief
Education
Youth
Religious Freedom
Interfaith
Cow Protection
Environment
Resources
ISKCON News
Media Relations
Videos
Resource Links
Academic Resources
Contact
Locate a Center
Contact Us
Chant. Dance. Feast. Serve.
Experience the joy of devotion
Experience Devotion
Chant. Dance. Feast. Serve.
Experience the joy of devotion
Experi

## Time to bring it together - the API for OpenAI is very simple!

In [12]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = gemini.chat.completions.create(
        model = model_gem,
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [13]:
summarize("https://iskcon.org/")

"Welcome to the official home of ISKCON, or as you probably know them, the folks in the saffron robes who’ve been trying to get you to chant and eat free food since 1966. Founded by A.C. Bhaktivedanta Swami Prabhupada—a guy who clearly had more hustle than the rest of us combined—this is the mothership for all things Hare Krishna.\n\nBasically, they’re peddling ancient Vedic wisdom, a metric fuck-ton of Bhakti-yoga, and the absolute conviction that God is a person who’s way more interesting than your average existential crisis. Between the kirtans, the Sunday feasts, and a suspiciously high level of dedication to cow protection, it’s a global community for anyone looking to ditch their shitty karma and trade it in for some spiritual enlightenment. If you’re into chanting, dancing like nobody's watching, and vegetarianism that isn't just sad, wilted lettuce, you’ve found the right cult—err, organization."

In [14]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [15]:
display_summary("https://iskcon.org/")

Welcome to the official home of ISKCON, the Hare Krishna crew that’s been turning people onto Vedic wisdom since 1966. Basically, if you’ve ever seen a bunch of guys in orange robes chanting in the street, you’ve found their fan club. 

Founded by A.C. Bhaktivedanta Swami Prabhupada—a guy who clearly had more hustle than the rest of us combined—this site is the ultimate encyclopedia for their brand of bhakti-yoga. They’ve got everything from the "how-to" on chanting, karma, and why they’re so obsessed with cows, to the lowdown on their global festivals and food relief programs. It’s a massive operation dedicated to the whole "Chant, Dance, Feast, Serve" lifestyle. If you’re looking to trade in your existential dread for some ancient scripture and a decent vegetarian meal, this is where the magic happens.

In [16]:
display_summary("https://cnn.com")

This is CNN’s digital dumping ground—a sprawling, chaotic archive of everything currently happening on this doomed planet. It’s essentially a massive buffet of global misery, celebrity gossip, and health tips for people who think they can outrun their own mortality with a decent salad. 

Between the endless sections covering wars, politics, and the inevitable heat death of the universe, you’ll find the real "meat" of the site: a never-ending, soul-crushing loop of ad-feedback forms. It’s a beautifully bloated machine designed to funnel you into a subscription while pretending to care about your technical difficulties with their garbage video player. If you’re looking to get overwhelmed by the sheer volume of human drama, or just want to report that their ads are, once again, ruining your entire existence, this is your holy land.

In [17]:
display_summary("https://anthropic.com")

Welcome to Anthropic, the self-proclaimed safe space for AI that’s desperately trying to ensure our future digital overlords don’t just incinerate us all for kicks. They’re a "public benefit corporation," which is fancy CEO-speak for "we’re trying to build god-tier intelligence without accidentally triggering a Terminator-style apocalypse." 

The site is a cluttered menu of research papers, policy manifestos—like the "Claude Constitution," which sounds like a pamphlet you’d find at a cult compound—and their lineup of Claude models. Whether you’re looking for a brainy chatbot to do your heavy lifting or just want to see which version of Claude is currently the least likely to hallucinate a lie, they’ve got a model for it. Basically, it’s the place to go if you want your AI served with a side of rigorous, pre-programmed "safety" protocols so you can get back to pretending you’re working while the machine does the actual heavy lifting.

In [18]:
display_summary("https://www.playboyclub.com/")

Listen, you technophobe. The website is essentially a digital bouncer holding a velvet rope, and you’re barred from entry because your browser is running on the computing equivalent of a hamster wheel. They’ve locked the entire Playboy Club experience behind a wall of JavaScript, so unless you flip that switch and join the modern era, you’re just staring at a blank screen like a total dipshit. Enable your scripts or keep enjoying the darkness. Your call.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise, you experienced calling the Cloud API of a Frontier Model (a leading model at the frontier of AI) for the first time. We will be using APIs like OpenAI at many stages in the course, in addition to building our own LLMs.

More specifically, we've applied this to Summarization - a classic Gen AI use case to make a summary. This can be applied to any business vertical - summarizing the news, summarizing financial performance, summarizing a resume in a cover letter - the applications are limitless. Consider how you could apply Summarization in your business, and try prototyping a solution.</span>
        </td>
    </tr>
</table>

